# Product Documentation Pipeline - ETL Notebook

This notebook provides the code for Snowflake ETL Pipeline

---
## 🔧 Setup & Configuration

Before running the pipeline, ensure you have:
1. A `.env` file with Snowflake credentials
2. The `INITIALS` environment variable set (e.g., `NS`)

### Install Dependencies

In [ ]:
# Uncomment to install dependencies
# !pip install -e .[dev,data]

In [1]:
# Standard imports
import os
import sys
from pathlib import Path
from datetime import datetime

# Ensure src is in the path
project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Load environment variables
try:
    from dotenv import load_dotenv
    load_dotenv()
    print("Environment loaded from .env")
except ImportError:
    print("python-dotenv not installed, using system environment variables")

Environment loaded from .env


In [3]:
# Import pipeline modules
from src.docs_pipeline.config import get_initials, tbl, get_snowflake_params, initials_is_valid
from src.docs_pipeline.snowflake import get_connection, test_connection
from src.docs_pipeline.ddl import (
    sitemap_staging_ddl, 
    docs_master_ddl, 
    document_content_ddl,
    pipeline_metrics_ddl,
    alerts_ddl
)

# Display configuration
initials = get_initials()
print(f"INITIALS: {initials}")
print(f"   Valid: {initials_is_valid(initials)}")
print()
print("Table Names:")
for table in ['SITEMAP_STAGING', 'DOCS_MASTER', 'DOCUMENT_CONTENT', 'PIPELINE_METRICS', 'ALERTS']:
    print(f"   • {tbl(table)}")

INITIALS: NS
   Valid: True

Table Names:
   • CANDIDATE_NS_SITEMAP_STAGING
   • CANDIDATE_NS_DOCS_MASTER
   • CANDIDATE_NS_DOCUMENT_CONTENT
   • CANDIDATE_NS_PIPELINE_METRICS
   • CANDIDATE_NS_ALERTS


---
## Test Snowflake Connection

In [4]:
# Test the Snowflake connection
result = test_connection()

if result.get('ok'):
    print("Snowflake connection: OK")
    print(f"   Version: {result['details'].get('version')}")
    print(f"   pandas available: {result['details'].get('pandas_available')}")
    print(f"   pyarrow available: {result['details'].get('pyarrow_available')}")
else:
    print("Snowflake connection: FAILED")
    print(f"   Error: {result.get('error')}")

Snowflake connection: OK
   Version: ('10.3.1',)
   pandas available: True
   pyarrow available: True


---
## Task 1: DDL - Table Definitions

The pipeline uses 5 tables with the naming convention `CANDIDATE_{INITIALS}_{TABLE_NAME}`:

1. **SITEMAP_STAGING** - Raw sitemap extraction data per run
2. **DOCS_MASTER** - Consolidated unique URLs with sources and timestamps
3. **DOCUMENT_CONTENT** - Fetched document content with metadata
4. **PIPELINE_METRICS** - Run-level observability metrics
5. **ALERTS** - Triggered alert records

In [9]:
# View all DDL statements
print("=" * 60)
print("SITEMAP_STAGING")
print("=" * 60)
print(sitemap_staging_ddl())

print("=" * 60)
print("DOCS_MASTER")
print("=" * 60)
print(docs_master_ddl())

print("=" * 60)
print("DOCUMENT_CONTENT")
print("=" * 60)
print(document_content_ddl())

print("=" * 60)
print("PIPELINE_METRICS")
print("=" * 60)
print(pipeline_metrics_ddl())

print("=" * 60)
print("ALERTS")
print("=" * 60)
print(alerts_ddl())

SITEMAP_STAGING

CREATE TABLE IF NOT EXISTS CANDIDATE_NS_SITEMAP_STAGING (
  RUN_ID             STRING      NOT NULL,
  SOURCE_IDENTIFIER  STRING      NOT NULL,
  SITEMAP_URL        STRING      NOT NULL,
  DOCUMENT_URL       STRING      NOT NULL,
  LASTMOD            TIMESTAMP_NTZ,
  EXTRACTED_AT       TIMESTAMP_NTZ NOT NULL DEFAULT CURRENT_TIMESTAMP(),
  CONSTRAINT PK_SITEMAP_STAGING PRIMARY KEY (RUN_ID, SOURCE_IDENTIFIER, DOCUMENT_URL)
);

DOCS_MASTER

CREATE TABLE IF NOT EXISTS CANDIDATE_NS_DOCS_MASTER (
  DOCUMENT_URL   STRING        NOT NULL,
  SOURCES        ARRAY         NOT NULL,
  FIRST_SEEN_AT  TIMESTAMP_NTZ NOT NULL,
  LAST_SEEN_AT   TIMESTAMP_NTZ NOT NULL,
  LASTMOD        TIMESTAMP_NTZ,
  UPDATED_AT     TIMESTAMP_NTZ NOT NULL DEFAULT CURRENT_TIMESTAMP(),
  CONSTRAINT PK_DOCS_MASTER PRIMARY KEY (DOCUMENT_URL)
);

DOCUMENT_CONTENT

CREATE TABLE IF NOT EXISTS CANDIDATE_NS_DOCUMENT_CONTENT (
  DOCUMENT_URL            STRING        NOT NULL,
  CONTENT_TEXT            STRING,
  

In [5]:
# Apply DDL to Snowflake
from src.docs_pipeline.snowflake import execute

conn = get_connection()
ddls = [
    sitemap_staging_ddl(),
    docs_master_ddl(),
    document_content_ddl(),
    pipeline_metrics_ddl(),
    alerts_ddl(),
]

for i, ddl in enumerate(ddls, 1):
    print(f"Applying DDL {i}/{len(ddls)}...")
    execute(conn, ddl)
    print("  -> OK")

conn.close()
print("\nAll tables created!")

Applying DDL 1/5...
  -> OK
Applying DDL 2/5...
  -> OK
Applying DDL 3/5...
  -> OK
Applying DDL 4/5...
  -> OK
Applying DDL 5/5...
  -> OK

All tables created!


---
##Task 1: Sitemap Extraction

The sitemap extractor:
- Follows the sitemap protocol (handles both `sitemapindex` and `urlset` types)
- Recursively crawls nested child sitemaps
- Normalizes URLs (strips fragments, trims whitespace)
- Supports gzipped sitemaps
- Batches inserts to Snowflake
### Architecture

In [14]:
# Display sitemap extraction module docstring and public functions
from src.docs_pipeline import sitemap_extract

print(sitemap_extract.__doc__)
print("\nKey Functions:")
print("  • extract_urls(starting: dict, max_urls=None) -> Iterator[Tuple]")
print("  • batch_rows(rows: Iterable, batch_size=500) -> Iterator[List]")
print("  • insert_sitemap_staging(conn, run_id, rows)")
print("  • run_sitemap_extract(run_id, sources, conn, ...)")

Sitemap extraction utilities (Task 1).

Features implemented:
- Parse sitemapindex vs urlset
- Recursive crawling of nested sitemaps with visited set
- Support for gzipped sitemaps (.xml.gz or gz content)
- URL normalization (strip fragments, trim whitespace)
- Batching helper to yield rows for insertion to staging

Public functions:
- extract_urls(starting: dict[str, str], max_urls=None)
- batch_rows(iterator, batch_size)

Note: DB write helpers are intentionally separate so tests can focus on parsing + crawling.


Key Functions:
  • extract_urls(starting: dict, max_urls=None) -> Iterator[Tuple]
  • batch_rows(rows: Iterable, batch_size=500) -> Iterator[List]
  • insert_sitemap_staging(conn, run_id, rows)
  • run_sitemap_extract(run_id, sources, conn, ...)


In [15]:
# Demo: Parse a sitemap locally (without DB write)
from src.docs_pipeline.sitemap_extract import extract_urls, batch_rows

# Example: Extract URLs from Snowflake docs sitemap (limit to 10 for demo)
demo_sources = {
    "docs": "https://docs.snowflake.com/en/sitemap.xml",
    "other-docs": "https://other-docs.snowflake.com/en/sitemap.xml"
}

print("Extracting URLs from sitemap (limited to 10 for demo)...\n")

count = 0
for source_id, sitemap_url, doc_url, lastmod in extract_urls(demo_sources, max_urls=10):
    count += 1
    print(f"{count}. [{source_id}] {doc_url}")
    print(f"   └── lastmod: {lastmod or '(none)'}")

print(f"\nExtracted {count} URLs")

Extracting URLs from sitemap (limited to 10 for demo)...

1. [docs] https://docs.snowflake.com/en/api-reference
   └── lastmod: (none)
2. [docs] https://docs.snowflake.com/en/appendices
   └── lastmod: (none)
3. [docs] https://docs.snowflake.com/en/collaboration/collaboration-listings-about
   └── lastmod: (none)
4. [docs] https://docs.snowflake.com/en/collaboration/collaboration-listings-legal
   └── lastmod: (none)
5. [docs] https://docs.snowflake.com/en/collaboration/collaboration-marketplace-about
   └── lastmod: (none)
6. [docs] https://docs.snowflake.com/en/collaboration/consumer-becoming
   └── lastmod: (none)
7. [docs] https://docs.snowflake.com/en/collaboration/consumer-listings-access
   └── lastmod: (none)
8. [docs] https://docs.snowflake.com/en/collaboration/consumer-listings-exploring
   └── lastmod: (none)
9. [docs] https://docs.snowflake.com/en/collaboration/consumer-listings-paying
   └── lastmod: (none)
10. [docs] https://docs.snowflake.com/en/collaboration/consumer-li

In [19]:
# Run full sitemap extraction

from src.docs_pipeline.sitemap_extract import run_sitemap_extract

run_id = f"notebook_run_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
sources = {
    "docs": "https://docs.snowflake.com/en/sitemap.xml",
    "other-docs": "https://other-docs.snowflake.com/en/sitemap.xml"
}

conn = get_connection()
total = run_sitemap_extract(
    run_id=run_id,
    sources=sources,
    conn=conn,
    batch_size=500
)
conn.close()

print(f"\nInserted {total} URLs into {tbl('SITEMAP_STAGING')} with run_id={run_id}")


Inserted 6620 URLs into CANDIDATE_NS_SITEMAP_STAGING with run_id=notebook_run_20260209_010511


---
## Task 2: Data Consolidation (MERGE)

The consolidation step:
- Uses a SQL MERGE to upsert data from `SITEMAP_STAGING` to `DOCS_MASTER`
- Maintains `FIRST_SEEN_AT` and `LAST_SEEN_AT` timestamps
- Unions the `SOURCES` array across runs
- Is fully **idempotent** - rerunning produces the same results

### The MERGE Query

In [20]:
# Display the MERGE SQL
from src.docs_pipeline.consolidate import _merge_sql

print("Consolidation MERGE SQL:")
print(_merge_sql())

Consolidation MERGE SQL:

MERGE INTO CANDIDATE_NS_DOCS_MASTER AS t
USING (
  SELECT
    DOCUMENT_URL,
    ARRAY_AGG(DISTINCT SOURCE_IDENTIFIER) AS SOURCES,
    MAX(LASTMOD) AS LASTMOD
  FROM CANDIDATE_NS_SITEMAP_STAGING
  WHERE RUN_ID = %(run_id)s
  GROUP BY DOCUMENT_URL
) AS s
ON t.DOCUMENT_URL = s.DOCUMENT_URL
WHEN MATCHED THEN UPDATE SET
  t.SOURCES = ARRAY_DISTINCT(ARRAY_CAT(t.SOURCES, s.SOURCES)),
  t.LAST_SEEN_AT = CURRENT_TIMESTAMP(),
  t.LASTMOD = COALESCE(GREATEST(t.LASTMOD, s.LASTMOD), t.LASTMOD, s.LASTMOD),
  t.UPDATED_AT = CURRENT_TIMESTAMP()
WHEN NOT MATCHED THEN INSERT (DOCUMENT_URL, SOURCES, FIRST_SEEN_AT, LAST_SEEN_AT, LASTMOD)
VALUES (s.DOCUMENT_URL, s.SOURCES, CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP(), s.LASTMOD);



In [23]:
# Run consolidation
# Requires a valid run_id that exists in SITEMAP_STAGING

from src.docs_pipeline.consolidate import consolidate_run

run_id = "notebook_run_20260209_005759"  # Replace with actual run_id

conn = get_connection()
summary = consolidate_run(conn, run_id)
conn.close()

print("Consolidation Summary:")
for key, value in summary.items():
    print(f"   {key}: {value}")

Consolidation Summary:
   run_id: notebook_run_20260209_005759
   inserted_from_staging: 6617
   docs_master_count_before: 6620
   docs_master_count_after: 6620


---
## Task 3: Content Fetching

The content fetcher:
- Implements **change detection** to avoid re-fetching unchanged content
- Uses three levels of optimization:
  1. **Sitemap LASTMOD** - Skip if source hasn't changed
  2. **Conditional GET** - Use ETag/If-Modified-Since headers
  3. **Hash comparison** - Compare content hashes as fallback
- Implements **throttling** (per-host delay)
- Implements **retries** with exponential backoff
- Respects `Retry-After` header

In [24]:
# Content fetching module overview
from src.docs_pipeline import content_fetch

print(content_fetch.__doc__)
print("\nConfiguration:")
print(f"   Default per-host delay: {content_fetch.DEFAULT_PER_HOST_DELAY}s")
print(f"   Default retries: {content_fetch.DEFAULT_RETRIES}")

Document content fetcher (Task 3).

Features:
- Throttling (simple per-host delay)
- Retries with exponential backoff and respect for Retry-After
- Conditional GET using ETag / If-Modified-Since
- NOT_MODIFIED handling (304)
- Hash compare fallback
- Write-back via provided `write_callback` (for testability)

Public API:
- fetch_url(session, url, headers) -> requests.Response
- compute_hash(bytes) -> hex string
- should_fetch_based_on_lastmod(doc_lastmod, content_last_success_at) -> bool
- fetch_and_process(session, doc_entry, content_entry, write_callback, settings) -> result dict
- run_content_fetch(conn, batch_size, settings)

Note: `doc_entry` is a dict-like with at least keys: DOCUMENT_URL, LASTMOD
`content_entry` is dict-like with keys from DOCUMENT_CONTENT table when present.
`write_callback` receives (document_url, update_dict) and is responsible for writing to DB.


Configuration:
   Default per-host delay: 0.1s
   Default retries: 3


In [25]:
# Demo: Change detection logic
from src.docs_pipeline.content_fetch import should_fetch_based_on_lastmod, compute_hash

print("Change Detection Demo:")
print()

# Case 1: Content is newer than last fetch
doc_lastmod = "2026-02-08T12:00:00"
last_success = "2026-02-01T12:00:00"
should_fetch = should_fetch_based_on_lastmod(doc_lastmod, last_success)
print(f"1. Doc lastmod: {doc_lastmod}, Last success: {last_success}")
print(f"   → Should fetch: {should_fetch} (content is newer)")
print()

# Case 2: Content is older than last fetch
doc_lastmod = "2026-01-01T12:00:00"
last_success = "2026-02-01T12:00:00"
should_fetch = should_fetch_based_on_lastmod(doc_lastmod, last_success)
print(f"2. Doc lastmod: {doc_lastmod}, Last success: {last_success}")
print(f"   → Should fetch: {should_fetch} (content unchanged, skip)")
print()

# Case 3: Missing lastmod
should_fetch = should_fetch_based_on_lastmod(None, last_success)
print(f"3. Doc lastmod: None, Last success: {last_success}")
print(f"   → Should fetch: {should_fetch} (no lastmod, must fetch)")
print()

# Hash computation demo
content = b"Hello, Snowflake!"
hash_value = compute_hash(content)
print(f"Content hash example:")
print(f"   Content: {content.decode()}")
print(f"   SHA256:  {hash_value}")

Change Detection Demo:

1. Doc lastmod: 2026-02-08T12:00:00, Last success: 2026-02-01T12:00:00
   → Should fetch: True (content is newer)

2. Doc lastmod: 2026-01-01T12:00:00, Last success: 2026-02-01T12:00:00
   → Should fetch: False (content unchanged, skip)

3. Doc lastmod: None, Last success: 2026-02-01T12:00:00
   → Should fetch: True (no lastmod, must fetch)

Content hash example:
   Content: Hello, Snowflake!
   SHA256:  864f985dd310f0a02736e252d4e8884d32be073dd2a848cd29ba2fd2ec21e26f


In [ ]:
# Run content fetch
from src.docs_pipeline.content_fetch import run_content_fetch_parallel

conn = get_connection()
summary = run_content_fetch_parallel(
    conn,
    batch_size=100,
    settings={'max_docs': 6620}  
)
conn.close()

print("Content Fetch Summary:")
print(f"   Total processed: {summary['total']}")
print(f"   Successes: {summary['successes']}")
print(f"   Failures: {summary['failures']}")

Progress: 100/6620 (1.5%)
Progress: 200/6620 (3.0%)
Progress: 300/6620 (4.5%)


---
## Task 4: Analytics Queries

The pipeline includes 5 baseline analytics queries:

| Query | Description |
|-------|-------------|
| **4a** | Document count grouped by source identifier |
| **4b** | Monthly document distribution (trailing 12 months) |
| **4c** | Content fetch success rate segmented by source |
| **4d** | Top 10 URL path segments |
| **4e** | Stale documents (LASTMOD >180 days old) |

In [ ]:
# Load and display the analytics queries
sql_path = project_root / "sql" / "task4_analytics.sql"

with open(sql_path, 'r') as f:
    sql_content = f.read()

print("Task 4 Analytics Queries:")
print("=" * 60)
print(sql_content)

In [ ]:
# Run analytics queries (uncomment to run)

from src.docs_pipeline.analytics import run_analytics

conn = get_connection()
results = run_analytics(conn)
conn.close()

for query_id, rows in sorted(results.items()):
    print(f"\n{'='*60}")
    print(f"Query {query_id}: {len(rows)} rows")
    print('='*60)
    for row in rows[:5]:  # Show first 5 rows
        print(row)
    if len(rows) > 5:
        print(f"... and {len(rows) - 5} more rows")

---
## Task 5: Query Optimization

Three optimization scenarios with cost vs. latency tradeoffs:

1. **7-day rolling window** - Cost-efficient (simple filter) vs Time-efficient (materialized view)
2. **URL count per source** - Approximate aggregates vs Exact parallel aggregates
3. **Content deduplication** - GROUP BY hash vs Window functions

In [ ]:
# Load and display the optimization queries
sql_path = project_root / "sql" / "task5_optimizations.sql"

with open(sql_path, 'r') as f:
    sql_content = f.read()

print("Task 5 Query Optimizations:")
print("=" * 60)
print(sql_content)

---
## Task 7: Observability & Alerting

The observability module provides:
- **PIPELINE_METRICS** - Run-level stats for each pipeline step
- **ALERTS** - Triggered alerts with severity levels (INFO/WARN/CRITICAL)

### Alert Conditions
- Failure rate >5% → WARN
- Failure rate >20% → CRITICAL
- No rows consolidated → WARN
- Empty extraction result → WARN

In [ ]:
# Observability module overview
from src.docs_pipeline import observability

print(observability.__doc__)
print("\nFunctions:")
print("  • record_pipeline_metrics(conn, run_id, pipeline_name, step_name, ...)")
print("  • raise_alert(conn, alert_id, run_id, alert_type, severity, message, ...)")

In [ ]:
# Query observability data

from src.docs_pipeline.snowflake import execute

conn = get_connection()

# Recent pipeline metrics
print("Recent Pipeline Metrics:")
metrics = execute(conn, f"""
    SELECT RUN_ID, PIPELINE_NAME, STEP_NAME, DURATION_SECONDS, SUCCESS_COUNT, FAILURE_COUNT
    FROM {tbl('PIPELINE_METRICS')}
    ORDER BY STARTED_AT DESC
    LIMIT 10
""")
for row in metrics or []:
    print(row)

# Recent alerts
print("\nRecent Alerts:")
alerts = execute(conn, f"""
    SELECT ALERT_ID, SEVERITY, ALERT_TYPE, MESSAGE, CREATED_AT
    FROM {tbl('ALERTS')}
    ORDER BY CREATED_AT DESC
    LIMIT 10
""")
for row in alerts or []:
    print(row)

conn.close()

---
## 📤 Task 8: Google Sheets Export

Export Task 4 analytics results to Google Sheets:
- Uses Google Sheets API v4
- Creates a separate tab for each query (4a, 4b, 4c, 4d, 4e)
- Requires service account credentials

In [ ]:
# Google Sheets export
# Requires GOOGLE_APPLICATION_CREDENTIALS env var or explicit credentials path

from src.docs_pipeline.export_sheets import export_to_sheets

spreadsheet_id = os.environ.get('GOOGLE_SHEETS_SPREADSHEET_ID', '')

conn = get_connection()
result = export_to_sheets(conn, spreadsheet_id=spreadsheet_id)
conn.close()

print("Export complete!")
print(f"   Spreadsheet ID: {result['spreadsheet_id']}")
print(f"   Sheets written: {result['sheets_written']}")

---
## Run Full Pipeline

Execute the entire pipeline end-to-end:
1. Extract sitemaps → SITEMAP_STAGING
2. Consolidate → DOCS_MASTER
3. Fetch content → DOCUMENT_CONTENT
4. Run analytics
5. Export to Google Sheets (optional)

In [ ]:
# Full pipeline run (uncomment to run)
from src.docs_pipeline.sitemap_extract import run_sitemap_extract
from src.docs_pipeline.consolidate import consolidate_run
from src.docs_pipeline.content_fetch import run_content_fetch
from src.docs_pipeline.analytics import run_analytics

run_id = f"full_run_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
sources = {
    "docs": "https://docs.snowflake.com/en/sitemap.xml"
}

print(f"Starting full pipeline run: {run_id}")
print("=" * 60)

# Step 1: Extract
print("\nStep 1: Sitemap Extraction")
conn = get_connection()
inserted = run_sitemap_extract(run_id, sources, conn=conn, max_urls=100)
print(f"   Inserted {inserted} URLs")

# Step 2: Consolidate
print("\nStep 2: Consolidation")
summary = consolidate_run(conn, run_id)
print(f"   Before: {summary['docs_master_count_before']} rows")
print(f"   After:  {summary['docs_master_count_after']} rows")

# Step 3: Content Fetch
print("\nStep 3: Content Fetch")
fetch_result = run_content_fetch(conn, settings={'max_docs': 50})
print(f"   Processed: {fetch_result['total']}")
print(f"   Successes: {fetch_result['successes']}")
print(f"   Failures:  {fetch_result['failures']}")

# Step 4: Analytics
print("\nStep 4: Analytics")
results = run_analytics(conn)
for qid, rows in sorted(results.items()):
    print(f"   {qid}: {len(rows)} rows")

conn.close()
print("\nPipeline complete!")

---
## Testing

Run the test suite to validate the pipeline implementation.

In [ ]:
# Run all tests
!cd /Users/hiruzen/Programming/Projects/Snoflake-Data-Intern-Assessment/ETL && python -m pytest tests/ -v --tb=short